> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAGzTIin49w/5zM403525G-teHXho4SLHg/view?utm_content=DAGzTIin49w&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=h29b78664e1)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install gradio==6.2.0 openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 beautifulsoup4==4.14.3

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. 文档切分

## 2.1 简介

文档切分是指将一个长文本的 Document 拆成若干个更小的段落（Chunks），每个段落大小适合被大模型理解和向量化处理。

## 2.2 切分方法

切分的方法有很多种，这里我们主要介绍两种：
- CharacterTextSplitter（基于字数进行划分）
- RecursiveCharacterTextSplitter（基于符号划分）

### 2.2.1 CharacterTextSplitter

比如这里有一段文档内容我们可以让该切分器进行切分：

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
some_text = """When writing documents, writers will use document structure to group content. This can convey to the reader, which idea's are related. For example, closely related ideas are in sentances. Similar ideas are in paragraphs. Paragraphs form a document. \n\n  Paragraphs are often delimited with a carriage return or two carriage returns. Carriage returns are the "backslash n" you see embedded in this string. Sentences have a period at the end, but also, have a space. and words are separated by space."""

此时可以设置一个字符切分器，然后每 450 个字符进行一次切分：

In [ ]:
c_splitter = CharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=0,
    separator = ' '
)
print(c_splitter.split_text(some_text))

### 2.2.2 RecursiveCharacterTextSplitter

RecursiveCharacterTextSplitter 会更加细致地分割文档，因为它不仅考虑分割后的文本长度，还会兼顾重叠字符。默认情况下， 其使用  ["\n\n", "\n", " ", ""]  四种特殊符号作为分割文本的标记，并按照优先级顺序进行分割。

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
some_text = """When writing documents, writers will use document structure to group content. This can convey to the reader, which idea's are related. For example, closely related ideas are in sentances. Similar ideas are in paragraphs. Paragraphs form a document. \n\n  Paragraphs are often delimited with a carriage return or two carriage returns. Carriage returns are the "backslash n" you see embedded in this string. Sentences have a period at the end, but also, have a space. and words are separated by space."""

In [ ]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=0, 
    separators=["\n\n", "\n", " ", ""]
)
print(r_splitter.split_text(some_text))

我们可以尝试真实的对前面网页里的内容进行切分：

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://zh.d2l.ai/chapter_introduction/index.html")
docs = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
 chunk_size = 1500,
 chunk_overlap = 150)
splits = text_splitter.split_documents(docs)
print(len(splits))

## 2.3 结构化文本切分器

### 2.3.1 Markdown 文档切分

由于 Markdown 本身就已经是“语义结构化文档”，里内部天然有：
- `# 章节`
- `## 小节`
- `### 子节`


比如现在我有一段 Markdown 文本，其内部有最多三级标题：

In [ ]:
markdown_document = "# Foo\n\n ## Bar\n\nHi this is Jim\n\nHi this is Joe\n\n ### Boo \n\n Hi this is Lance \n\n ## Baz\n\n Hi this is Molly"

此时假如要对该文本进行切分，我们需要先设置三级标题，并且给出对应的名称：

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

然后就可以载入 MarkdownHeaderTextSplitter （需要安装依赖 langchain-text-splitters），并将该段文本切分：

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)
print(md_header_splits)

假如希望对 page_content 里的内容进行更进一步的划分，也可以在使用完 Mardown 拆分器后再使用 RecursiveCharacterTextSplitter 进行切分。

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=20, chunk_overlap=5)
splits = text_splitter.split_documents(md_header_splits)
print(splits)  

### 2.3.2 HTML 文档切分

对于 HTML 里的文档切分其实也是类似的，但是不一样的是 HTML 不只是“文本”，而是“带语义结构的文档树（DOM）”。 HTML 里有：

- 标题层级（h1 ~ h6）
- 段落（p）
- 列表（ul / ol）
- 表格（table）
- 媒体（img / video / iframe）
- 代码块（pre / code）

这也是为什么假如我们只按字符切，语义会被直接破坏。所以 HTML 切分的核心目标是在“可检索长度”和“语义完整性”之间取得平衡。

### 2.3.3 JSON 文档切分
对于 JSON 文档而言，LangChain 中提供了 RecursiveJsonSplitter 方法进行切割，其按 JSON 结构递归切分大型 JSON 对象，尽量保持嵌套结构完整，并控制每一块的字符大小。

它的切分逻辑和“按段落切文本”完全不同，是结构优先，而不是字符串优先：
- 深度优先遍历 JSON
- 优先保持一个对象（dict）不被拆散
- 不会随意切字符串
- 不默认拆 list（因为 list 通常是语义整体）

### 2.3.4 代码文档切分
对于代码类文件，LangChain 中也有相关的切分工具，其标不是“按长度切”，而是尽量按“代码结构边界”切，避免把一个函数、类或语句块拆碎。

这里支持的语言包括 python, cpp, go, java, rust, html, latex, markdown, c 等，这些切分的规则可以通过 Language 模块进行获取：

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)

在实际切分时，可以设置对应的语言即可（更多其他语言切分示例请查阅文档内容）：

In [ ]:
from langchain_text_splitters import (RecursiveCharacterTextSplitter, Language)

PYTHON_CODE = """
def hello_world():
    print("Hello, World!")

# Call the function
hello_world()
"""

python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0)
python_docs = python_splitter.create_documents([PYTHON_CODE])
print(python_docs)